# VNA Stage A — Phase-0 checkpoints, T4×2
Attach Phase-0 notebook outputs (extracted recipe or tar/zip) and CIFAR-100.
Phase-0 checkpoints are discovered automatically. Enable Internet, T4×2, github_token.
Only diagnostics run. No pytest is run. Immutable REPO_COMMIT must
include Stage A; regenerate with --commit after making the code available remotely.
GPU0 runs A5 at epochs 90, 100, 110, 160, 200 first, then the two-checkpoint cost smoke, then A1 and A3.
The smoke prints projected GPU-hours and rejects totals above 3 unless explicitly allowed. GPU1 runs A4 then A2 then A6. Completion markers resume
per measurement/epoch/site (selection also rank/kind) with schema/config/checkpoint-hash validation. Hourly
archives use the existing notebook disk cap. Attach prior Stage-A outputs to resume.


In [ ]:
from pathlib import Path
import time
REPO_COMMIT = "98ad44d381e0a3af06116f21a30bfcf6fc14ce39"  # immutable, builder --commit; default current HEAD
GPU_HOURS_BUDGET = 3
ALLOW_OVER_BUDGET = False
SESSION_HOURS = 11.5
SESSION_START = time.time()
VNA_DEADLINE = SESSION_START + SESSION_HOURS * 3600


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
REFERENCE_URL = "https://github.com/duyh80456-code/One-Shot-TAS-CCIL.git"
REFERENCE = Path("/kaggle/working/One-Shot-TAS-CCIL")
GROMO_URL = "https://github.com/growingnet/gromo.git"
GROMO_COMMIT = "8d19107b61a9459a9021065a329b699adcb0f25b"
GROMO = Path("/kaggle/working/gromo")
RUN_ROOT = Path("/kaggle/working/vna_stage_a")
OUTPUT = RUN_ROOT

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        if branch == REPO_COMMIT:
            subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                            url, str(destination)], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "fetch", "origin", REPO_COMMIT], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "checkout", "--detach", REPO_COMMIT], check=True)
        else:
            subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                            url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

for checkout in (REPO, REFERENCE, GROMO):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, REPO_COMMIT)
private_clone(REFERENCE_URL, REFERENCE, "ccil-residual-capacity")
subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                GROMO_URL, str(GROMO)], check=True)
subprocess.run(["git", "-C", str(GROMO), "fetch", "--depth", "1",
                "origin", GROMO_COMMIT], check=True)
subprocess.run(["git", "-C", str(GROMO), "checkout", "--detach",
                GROMO_COMMIT], check=True)
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

if subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() != REPO_COMMIT:
    raise RuntimeError("Pinned repository revision mismatch")

from datetime import datetime, timezone
CODE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Resolved code commit:", CODE_COMMIT, "pin:", REPO_COMMIT)
commit_path = OUTPUT / "code_commit.json"
sessions = json.loads(commit_path.read_text()) if commit_path.exists() else []
sessions.append({"timestamp": datetime.now(timezone.utc).isoformat(), "pin": REPO_COMMIT, "commit": CODE_COMMIT})
temporary = commit_path.with_suffix(".json.tmp")
temporary.write_text(json.dumps(sessions, indent=2) + "\n")
temporary.replace(commit_path)


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[test,vision,diagnostics]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(GROMO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-upstream.txt")], check=True)
subprocess.run(["bash", "scripts/fetch_third_party.sh"], cwd=REPO, check=True)
GROMO_SRC = GROMO / "src"
RUNTIME_PYTHONPATH = os.pathsep.join(filter(None, (
    str(REPO), str(GROMO_SRC), str(REFERENCE), os.environ.get("PYTHONPATH", ""))))
test_env = os.environ.copy(); test_env.update(PYTHONPATH=RUNTIME_PYTHONPATH, REQUIRE_UPSTREAM="1", PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION="python")
import torch
if torch.cuda.device_count() != 2:
    raise RuntimeError(f"Select T4 x2; found {torch.cuda.device_count()} GPU(s)")
required_cifar_files = {"train", "test", "meta"}
cifar_dirs = []
for directory, _, filenames in os.walk("/kaggle/input", followlinks=True):
    path = Path(directory)
    if (path.name == "cifar-100-python" and
            required_cifar_files.issubset(filenames)):
        cifar_dirs.append(path.parent)
cifar_dirs = sorted(set(cifar_dirs), key=str)
if not cifar_dirs:
    visible = []
    for directory, dirnames, filenames in os.walk(
            "/kaggle/input", followlinks=True):
        visible.append({
            "directory": directory,
            "directories": sorted(dirnames)[:10],
            "files": sorted(filenames)[:10],
        })
        if len(visible) == 20:
            break
    raise FileNotFoundError(
        "Attach CIFAR-100 containing cifar-100-python/{train,test,meta}. "
        f"Visible input inventory: {visible}")
DATA_ROOT = cifar_dirs[0]
print("CIFAR-100 root:", DATA_ROOT)

if not (REPO / "experiments/vna_kaggle.py").exists():
    raise RuntimeError("Required module experiments/vna_kaggle.py is missing from the pinned commit")
for module in ("experiments/vna_provenance.py", "diagnostics/signal_curve.py", "methods/vna.py"):
    if not (REPO / module).exists(): raise RuntimeError(f"Required module {module} is missing from the pinned commit")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=True)
sys.path[:0] = [str(REPO), str(GROMO_SRC), str(REFERENCE)]
print("Official TEST verified:", DATA_ROOT / "cifar-100-python/test")

for module in ("diagnostics/stage_a.py", "methods/vna_joint.py", "diagnostics/vna_gate.py"):
    if not (REPO / module).exists(): raise RuntimeError(f"Missing Stage-A module: {module}")


In [ ]:
from experiments.vna_kaggle import archive_with_status as _archive_outputs
from diagnostics.stage_a import restore_archives
import shutil
def archive_outputs(output):
    return _archive_outputs(output, folders=("outputs/stageA",))
STAGE_A = RUN_ROOT / "outputs" / "stageA"
STAGE_A.mkdir(parents=True, exist_ok=True)
from experiments.phase0_discovery import discover_phase0
RECIPE = discover_phase0()
# Restore Stage-A directory and tar/zip outputs from prior sessions.
restore_archives(Path("/kaggle/input"), STAGE_A)
common = ["--checkpoint-dir", str(RECIPE), "--data-root", str(DATA_ROOT),
          "--output", str(STAGE_A), "--seed", "0"]
ENV = {**test_env, "PYTHONUNBUFFERED": "1", "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2", "OPENBLAS_NUM_THREADS": "2", "VNA_DEADLINE": str(VNA_DEADLINE)}
archive_stop = threading.Event()
archive_errors = []
def periodic_archive():
    while not archive_stop.wait(3600):
        try: archive_outputs(RUN_ROOT)
        except Exception as error:
            archive_errors.append(str(error)); print("Archive error:", error, flush=True)
archiver = threading.Thread(target=periodic_archive, daemon=True)
archiver.start()
session_paused = threading.Event()
lane_state = {gpu: {"job": "idle", "last": ""} for gpu in range(2)}
def pause_due():
    if time.time() >= VNA_DEADLINE - 600: session_paused.set()
    return session_paused.is_set()
def launch(cmd, gpu, name, log_path):
    from experiments.vna_resume import launch_logged
    return launch_logged(cmd, cwd=REPO, env=gpu_env(gpu), log_path=log_path,
                         lane=gpu, job=name, lanes=lane_state,
                         paused=session_paused, deadline=VNA_DEADLINE)
def heartbeat():
    from experiments.vna_resume import print_heartbeat
    while not archive_stop.wait(600):
        print_heartbeat(RUN_ROOT / "outputs", SESSION_START, VNA_DEADLINE, lane_state)

def gpu_env(gpu): return {**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}
heartbeat_thread = threading.Thread(target=heartbeat, daemon=True); heartbeat_thread.start()
def stage_launch(measurement, gpu=0, extra=()):
    return launch([sys.executable, "-u", "-m", "diagnostics.stage_a", measurement,
                   *common, "--device", "cuda:0", *extra], gpu, "stageA_"+measurement,
                  RUN_ROOT / f"{measurement}_gpu{gpu}.log")


In [ ]:
try:
    if stage_launch("cost", extra=("--checkpoints",
            str(RECIPE / "epoch_090.pt"), str(RECIPE / "epoch_100.pt"))):
        cost = json.loads((STAGE_A / "cost_projection.json").read_text())
        hours = cost["total_projected_gpu_hours"] + cost["smoke_gpu_hours"]
        print("Stage A projected GPU-hours including smoke:", hours, flush=True)
        if hours > GPU_HOURS_BUDGET and not ALLOW_OVER_BUDGET:
            raise RuntimeError("Stage A exceeds GPU_HOURS_BUDGET")
    if not session_paused.is_set(): stage_launch("a5")
    archive_outputs(RUN_ROOT)
except BaseException:
    archive_stop.set(); archiver.join(timeout=10); archive_outputs(RUN_ROOT); raise


In [ ]:
from concurrent.futures import ThreadPoolExecutor
def worker(gpu, measurements):
    for measurement in measurements:
        if not stage_launch(measurement, gpu): break
try:
    with ThreadPoolExecutor(max_workers=2) as pool:
        futures = [pool.submit(worker, 0, ["null", "selection"]),
                   pool.submit(worker, 1, ["joint", "mu", "solver"])]
        for future in futures: future.result()
    if not session_paused.is_set(): stage_launch("summary")
finally:
    archive_stop.set(); archiver.join(timeout=10); heartbeat_thread.join(timeout=2)
    archive_outputs(RUN_ROOT)
if session_paused.is_set(): print("PAUSED — rerun with this version's output attached", flush=True)
if archive_errors: raise RuntimeError(archive_errors)


In [ ]:
if not session_paused.is_set():
    import pandas as pd
    import matplotlib.pyplot as plt
    print((STAGE_A / "summary.json").read_text())
    for filename in ("a5.csv", "null.csv", "mu_sweep.csv", "selection.csv", "joint.csv", "solver.csv"):
        print(filename)
        display(pd.read_csv(STAGE_A / filename))
    mu = pd.read_csv(STAGE_A / "mu_sweep.csv")
    fig, ax = plt.subplots()
    for (epoch, site), group in mu.groupby(["epoch", "site"]):
        ax.plot(group.mu_rel, group.rho, marker="o", label=f"{epoch} {site}")
    ax.set_xscale("log"); ax.set(xlabel="mu_rel", ylabel="rho")
    fig.savefig(STAGE_A / "rho_mu.png", bbox_inches="tight"); plt.show()
    selection = pd.read_csv(STAGE_A / "selection.csv")
    selection.groupby(["rank", "kind"]).V.mean().unstack().plot.bar(ylabel="V")
    plt.tight_layout(); plt.savefig(STAGE_A / "selection_V.png"); plt.show()
    null = pd.read_csv(STAGE_A / "null.csv")
    null.boxplot(column="p_value", by=["null_kind", "stat"])
    plt.axhline(1/9, color="red"); plt.tight_layout()
    plt.savefig(STAGE_A / "null_p_values.png"); plt.show()
    archive_outputs(RUN_ROOT)
    print("Archive:", RUN_ROOT.with_suffix(".tar.gz"), "includes outputs/stageA/")
